## Why We Need Batch Normalization

Batch Normalization normalizes each layer's activations using mini-batch
mean/variance, then rescales with two learnable parameters. Below, a deep
MLP and a deep CNN are each trained identically with and without it
(`BatchNorm1d`/`BatchNorm2d`) -- the plots and printed stats make the case.
FashionMNIST, CPU, each experiment runs in well under a minute.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [ ]:
torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cpu")
print("device:", device)

### 1. Dataset: FashionMNIST

28x28 grayscale, 10 classes, 60k train / 10k test images.

In [ ]:
# Download the raw Fashion-MNIST archives directly from the source mirror into data/,
# before loading them with torchvision. wget's -nc (no-clobber) flag skips the download
# if a file is already present, so re-running this cell will not re-download the data.
DATA_URL = "http://fashion-mnist.s3-website.eu-central-1.amazonaws.com"
RAW_DIR = "data/FashionMNIST/raw"

!mkdir -p {RAW_DIR}
!wget -nc -nv -P {RAW_DIR} {DATA_URL}/train-images-idx3-ubyte.gz
!wget -nc -nv -P {RAW_DIR} {DATA_URL}/train-labels-idx1-ubyte.gz
!wget -nc -nv -P {RAW_DIR} {DATA_URL}/t10k-images-idx3-ubyte.gz
!wget -nc -nv -P {RAW_DIR} {DATA_URL}/t10k-labels-idx1-ubyte.gz

In [ ]:
transform = transforms.ToTensor()  # scales pixels to [0, 1], shape (1, 28, 28)

# download=True only extracts here: the archives were already fetched by wget above
train_ds = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
test_ds = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

print(f"train images: {len(train_ds)}, test images: {len(test_ds)}, image shape: {train_ds[0][0].shape}")

### 2. A deep MLP, with a batch norm switch

8 hidden blocks of `Linear -> [BatchNorm] -> ReLU`. `use_bn=False` swaps
`BatchNorm1d` for `nn.Identity`, so the two networks differ only in that
switch. A hook records one middle layer's output (`last_activation`) for the
distribution plots below.

In [ ]:
class DeepMLP(nn.Module):
    def __init__(self, use_bn, depth=8, width=128, track_layer=4):
        super().__init__()
        self.track_layer = track_layer
        self.flatten = nn.Flatten()
        self.relu = nn.ReLU()
        self.blocks = nn.ModuleList()
        in_features = 28 * 28
        for _ in range(depth):
            self.blocks.append(nn.ModuleDict({
                "linear": nn.Linear(in_features, width),
                "bn": nn.BatchNorm1d(width) if use_bn else nn.Identity(),
            }))
            in_features = width
        self.out = nn.Linear(width, 10)
        self.last_activation = None  # (batch, width) output of the tracked layer, post-BN/pre-ReLU

    def forward(self, x):
        x = self.flatten(x)
        for i, block in enumerate(self.blocks):
            x = block["bn"](block["linear"](x))
            if i == self.track_layer:
                self.last_activation = x.detach()
            x = self.relu(x)
        return self.out(x)

### 3. Training utilities

Plain mini-batch SGD, no momentum, to isolate batch norm's effect. 

Records loss and the tracked layer's mean/std every step; 
stops early if a run diverges.

In [ ]:
def train_model(model, loader, lr, epochs):
    model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {"step_loss": [], "act_mean": [], "act_std": [], "epoch_acc": []}
    for epoch in range(epochs):
        model.train()
        correct, total = 0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)

            if torch.isnan(loss) or loss.item() > 50:
                print(f"  epoch {epoch + 1}: diverged (loss={loss.item():.2f}), stopping run early")
                return history

            loss.backward()
            optimizer.step()

            history["step_loss"].append(loss.item())
            history["act_mean"].append(model.last_activation.mean().item())
            history["act_std"].append(model.last_activation.std().item())
            correct += (out.argmax(1) == yb).sum().item()
            total += yb.size(0)

        epoch_acc = correct / total
        history["epoch_acc"].append(epoch_acc)
        print(f"  epoch {epoch + 1}/{epochs}  train_acc={epoch_acc:.4f}")
    return history


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        correct += (out.argmax(1) == yb).sum().item()
        total += yb.size(0)
    return correct / total

### 4. Experiment 1: training dynamics at lr=0.1

Same seed, data order, and learning rate, only the batch norm switch differs.
`use_bn=False | True`

In [ ]:
EPOCHS = 3
LR = 0.1

print("Training deep MLP WITHOUT batch norm...")
torch.manual_seed(0)
model_no_bn = DeepMLP(use_bn=False)
hist_no_bn = train_model(model_no_bn, train_loader, lr=LR, epochs=EPOCHS)
acc_no_bn = evaluate(model_no_bn, test_loader)
print(f"test accuracy: {acc_no_bn:.4f}\n")


print("Training deep MLP WITH batch norm...")
torch.manual_seed(0)
model_bn = DeepMLP(use_bn=True)
hist_bn = train_model(model_bn, train_loader, lr=LR, epochs=EPOCHS)
acc_bn = evaluate(model_bn, test_loader)
print(f"test accuracy: {acc_bn:.4f}")

In [ ]:
COLOR_NO_BN = "#2a78d6"   # blue
COLOR_BN = "#eb6834"      # orange

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=False)

ax = axes[0]
ax.plot(hist_no_bn["step_loss"], color=COLOR_NO_BN, linewidth=1, label="without batch norm")
ax.plot(hist_bn["step_loss"], color=COLOR_BN, linewidth=1, label="with batch norm")
ax.set_xlabel("training step")
ax.set_ylabel("training loss")
ax.set_title(f"Training loss per step (lr={LR})")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
epochs_x = range(1, EPOCHS + 1)
ax.plot(epochs_x, hist_no_bn["epoch_acc"], color=COLOR_NO_BN, marker="o", linewidth=2, label="without batch norm")
ax.plot(epochs_x, hist_bn["epoch_acc"], color=COLOR_BN, marker="o", linewidth=2, label="with batch norm")
ax.set_xlabel("epoch")
ax.set_ylabel("train accuracy")
ax.set_title("Train accuracy per epoch")
ax.set_xticks(list(epochs_x))
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

print(f"final test accuracy -- without BN: {acc_no_bn:.4f}   with BN: {acc_bn:.4f}")

Without batch norm the network is still near random guessing after 2
epochs; with it, learning is well underway from epoch 1 (final numbers
printed above).

### 5. What batch norm is controlling: the tracked layer's output

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

ax = axes[0]
ax.plot(hist_no_bn["act_mean"], color=COLOR_NO_BN, linewidth=1, label="without batch norm")
ax.plot(hist_bn["act_mean"], color=COLOR_BN, linewidth=1, label="with batch norm")
ax.set_ylabel("layer output mean")
ax.set_title("Middle-layer output distribution during training")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
ax.plot(hist_no_bn["act_std"], color=COLOR_NO_BN, linewidth=1, label="without batch norm")
ax.plot(hist_bn["act_std"], color=COLOR_BN, linewidth=1, label="with batch norm")
ax.set_xlabel("training step")
ax.set_ylabel("layer output std")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

Without batch norm the output stays collapsed near zero until the network
breaks through around step 1200 -- matching the plateau above. With batch
norm it's anchored at mean 0 / std 1 from step one.

- Layer Output Mean: For the models with batch normalization (the orange lines), the mean of the middle layer's output remains consistently close to zero throughout the training steps. In contrast, for models without batch normalization (the blue lines), the mean fluctuates significantly, sometimes deviating far from zero, indicating unstable activation distributions.

- Layer Output Standard Deviation: Similarly, for the models with batch normalization, the standard deviation of the middle layer's output stays consistently close to one. This is a direct effect of batch normalization, which scales activations to have unit variance. Conversely, the standard deviation for models without batch normalization shows large and erratic changes, often collapsing near zero initially and then exploding to very large values. This suggests that without batch normalization, the scale of activations is highly unstable.

**Interpretation:** Batch Normalization explicitly re-centers the activations to have a mean of approximately zero and re-scales them to have a standard deviation of approximately one. This keeps the distribution of activations stable, preventing the 'internal covariate shift' problem where changes in previous layers' parameters can lead to significant changes in the distribution of inputs to subsequent layers. This stability makes the training process much smoother, allows for higher learning rates, and helps prevent vanishing or exploding gradients, ultimately leading to faster convergence and better performance, as seen in the accuracy plots.

### 6. Experiment 2: learning-rate sweep

Final test accuracy vs. learning rate, 2 epochs each, both networks.

In [ ]:
LRS = [0.01, 0.03, 0.1, 0.3, 1.0]
SWEEP_EPOCHS = 2

acc_no_bn_by_lr, acc_bn_by_lr = [], []
for lr in LRS:
    print(f"lr={lr}")

    torch.manual_seed(0)
    m = DeepMLP(use_bn=False)
    print(" without batch norm:")
    train_model(m, train_loader, lr=lr, epochs=SWEEP_EPOCHS)
    acc_no_bn_by_lr.append(evaluate(m, test_loader))

    torch.manual_seed(0)
    m = DeepMLP(use_bn=True)
    print(" with batch norm:")
    train_model(m, train_loader, lr=lr, epochs=SWEEP_EPOCHS)
    acc_bn_by_lr.append(evaluate(m, test_loader))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(LRS, acc_no_bn_by_lr, color=COLOR_NO_BN, marker="o", linewidth=2, label="without batch norm")
ax.plot(LRS, acc_bn_by_lr, color=COLOR_BN, marker="o", linewidth=2, label="with batch norm")
ax.set_xscale("log")
ax.set_xlabel("learning rate (log scale)")
ax.set_ylabel("test accuracy")
ax.set_title(f"Test accuracy vs. learning rate ({SWEEP_EPOCHS} epochs)")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

Batch norm trains well across the whole sweep. The plain network only
escapes random guessing in a narrow band around `lr=0.3`, and diverges
beyond it (numbers above).

## Part 2: the same story on a CNN

`nn.BatchNorm2d` normalizes each channel over the batch *and* spatial
dimensions. Same two experiments, now on a deep CNN.

### 7. A deep CNN, with the same switch

6 conv layers, `Conv2d -> [BatchNorm2d] -> ReLU`, 3 of them strided
(28->14->7->4), then global average pooling and a linear head. Same
`use_bn` switch and activation hook as the MLP.

In [ ]:
class DeepCNN(nn.Module):
    def __init__(self, use_bn, depth=6, width=32, downsample_at=(1, 3, 5), track_layer=3):
        super().__init__()
        self.track_layer = track_layer
        self.blocks = nn.ModuleList()
        in_channels = 1
        for i in range(depth):
            stride = 2 if i in downsample_at else 1
            self.blocks.append(nn.ModuleDict({
                "conv": nn.Conv2d(in_channels, width, kernel_size=3, stride=stride, padding=1),
                "bn": nn.BatchNorm2d(width) if use_bn else nn.Identity(),
            }))
            in_channels = width
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out = nn.Linear(width, 10)
        self.last_activation = None  # (batch, width, H, W) output of the tracked layer

    def forward(self, x):
        for i, block in enumerate(self.blocks):
            x = block["bn"](block["conv"](x))
            if i == self.track_layer:
                self.last_activation = x.detach()
            x = self.relu(x)
        x = self.pool(x).flatten(1)
        return self.out(x)

### 8. Experiment 3: CNN training dynamics at lr=0.1

In [ ]:
torch.manual_seed(0)
cnn_no_bn = DeepCNN(use_bn=False)
print("Training deep CNN WITHOUT batch norm...")
cnn_hist_no_bn = train_model(cnn_no_bn, train_loader, lr=LR, epochs=EPOCHS)
cnn_acc_no_bn = evaluate(cnn_no_bn, test_loader)
print(f"test accuracy: {cnn_acc_no_bn:.4f}\n")

torch.manual_seed(0)
cnn_bn = DeepCNN(use_bn=True)
print("Training deep CNN WITH batch norm...")
cnn_hist_bn = train_model(cnn_bn, train_loader, lr=LR, epochs=EPOCHS)
cnn_acc_bn = evaluate(cnn_bn, test_loader)
print(f"test accuracy: {cnn_acc_bn:.4f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7))

ax = axes[0]
ax.plot(cnn_hist_no_bn["step_loss"], color=COLOR_NO_BN, linewidth=1, label="without batch norm")
ax.plot(cnn_hist_bn["step_loss"], color=COLOR_BN, linewidth=1, label="with batch norm")
ax.set_xlabel("training step")
ax.set_ylabel("training loss")
ax.set_title(f"CNN training loss per step (lr={LR})")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
epochs_x = range(1, EPOCHS + 1)
ax.plot(epochs_x, cnn_hist_no_bn["epoch_acc"], color=COLOR_NO_BN, marker="o", linewidth=2, label="without batch norm")
ax.plot(epochs_x, cnn_hist_bn["epoch_acc"], color=COLOR_BN, marker="o", linewidth=2, label="with batch norm")
ax.set_xlabel("epoch")
ax.set_ylabel("train accuracy")
ax.set_title("CNN train accuracy per epoch")
ax.set_xticks(list(epochs_x))
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

print(f"final test accuracy -- without BN: {cnn_acc_no_bn:.4f}   with BN: {cnn_acc_bn:.4f}")

Same pattern as the MLP: stuck near random guessing until a late
breakthrough without batch norm; strong accuracy from epoch 1 with it
(numbers above).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

ax = axes[0]
ax.plot(cnn_hist_no_bn["act_mean"], color=COLOR_NO_BN, linewidth=1, label="without batch norm")
ax.plot(cnn_hist_bn["act_mean"], color=COLOR_BN, linewidth=1, label="with batch norm")
ax.set_ylabel("layer output mean")
ax.set_title("CNN middle-layer output distribution during training")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
ax.plot(cnn_hist_no_bn["act_std"], color=COLOR_NO_BN, linewidth=1, label="without batch norm")
ax.plot(cnn_hist_bn["act_std"], color=COLOR_BN, linewidth=1, label="with batch norm")
ax.set_xlabel("training step")
ax.set_ylabel("layer output std")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

Without batch norm the layer collapses near zero, then overshoots past std
> 10 once gradients finally flow. With batch norm it stays anchored at mean
0 / std 1 throughout.

### 9. Experiment 4: CNN learning-rate sweep

Same five learning rates as Experiment 2, 2 epochs each, on a
12,800-image subset (conv passes are heavier -- this keeps the sweep
quick).

In [ ]:
from torch.utils.data import Subset

sweep_loader = DataLoader(Subset(train_ds, range(12800)), batch_size=128, shuffle=True)

cnn_acc_no_bn_by_lr, cnn_acc_bn_by_lr = [], []
for lr in LRS:
    print(f"lr={lr}")

    torch.manual_seed(0)
    m = DeepCNN(use_bn=False)
    print(" without batch norm:")
    train_model(m, sweep_loader, lr=lr, epochs=SWEEP_EPOCHS)
    cnn_acc_no_bn_by_lr.append(evaluate(m, test_loader))

    torch.manual_seed(0)
    m = DeepCNN(use_bn=True)
    print(" with batch norm:")
    train_model(m, sweep_loader, lr=lr, epochs=SWEEP_EPOCHS)
    cnn_acc_bn_by_lr.append(evaluate(m, test_loader))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(LRS, cnn_acc_no_bn_by_lr, color=COLOR_NO_BN, marker="o", linewidth=2, label="without batch norm")
ax.plot(LRS, cnn_acc_bn_by_lr, color=COLOR_BN, marker="o", linewidth=2, label="with batch norm")
ax.set_xscale("log")
ax.set_xlabel("learning rate (log scale)")
ax.set_ylabel("test accuracy")
ax.set_title(f"CNN test accuracy vs. learning rate ({SWEEP_EPOCHS} epochs, 12,800-image subset)")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

Without batch norm the CNN doesn't learn at any of these learning rates.
With it, accuracy holds up across the entire range (numbers above).

In [ ]:
import numpy as np

archs = ["MLP", "CNN"]
fixed_lr_no_bn = [acc_no_bn, cnn_acc_no_bn]
fixed_lr_bn = [acc_bn, cnn_acc_bn]
sweep_no_bn = [np.mean(acc_no_bn_by_lr), np.mean(cnn_acc_no_bn_by_lr)]
sweep_bn = [np.mean(acc_bn_by_lr), np.mean(cnn_acc_bn_by_lr)]

x = np.arange(len(archs))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

ax = axes[0]
ax.bar(x - width / 2, fixed_lr_no_bn, width, color=COLOR_NO_BN, label="without batch norm")
ax.bar(x + width / 2, fixed_lr_bn, width, color=COLOR_BN, label="with batch norm")
ax.set_xticks(x)
ax.set_xticklabels(archs)
ax.set_ylabel("test accuracy")
ax.set_title("Fixed lr=0.1 (Experiments 1, 3)")
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
ax.bar(x - width / 2, sweep_no_bn, width, color=COLOR_NO_BN, label="without batch norm")
ax.bar(x + width / 2, sweep_bn, width, color=COLOR_BN, label="with batch norm")
ax.set_xticks(x)
ax.set_xticklabels(archs)
ax.set_ylabel("mean test accuracy over LR sweep")
ax.set_title("Averaged over the sweep (Experiments 2, 4)")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Batch norm, summarized")
fig.tight_layout()
plt.show()

### Conclusion

- **Faster convergence** -- loss drops sooner and more smoothly (Experiments 1, 3).
- **Stable layer inputs** -- the tracked layer stays near mean 0 / std 1
  instead of collapsing or blowing up (the distribution plots).
- **A much wider working range of learning rates** -- no narrow sweet spot
  to search for (Experiments 2, 4).

Same switch, same benefit, in both a plain MLP and a CNN.